# MGMT298D: Science and Strategy of AI
## Week 1: Linear Regression & Regularization
### UCLA Anderson School of Management

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_style('whitegrid')

## Load Data

In [ ]:
# Download H&M sales data from GitHub
url = "https://raw.githubusercontent.com/ucla-anderson-SSAI/SSAI/main/HMData.csv"
df = pd.read_csv(url)
df_product1 = df[df['name'] == df['name'].unique()[0]].reset_index(drop=True)

print(f"Shape: {df_product1.shape}")
print(f"\nFirst few rows:")
df_product1.head()

## Feature Engineering

In [ ]:
# Create lag features, moving average, and price change
df_product1['lag_m1'] = df_product1['sales'].shift(1)
df_product1['lag_m2'] = df_product1['sales'].shift(2)
df_product1['ma_3'] = df_product1['sales'].rolling(window=3).mean()
df_product1['price_change'] = df_product1['price'].diff()

# Fill NaN values with 0
df_product1.fillna(0, inplace=True)

# Display descriptive statistics
print("Descriptive Statistics:")
df_product1[['sales', 'price', 'lag_m1', 'lag_m2', 'ma_3', 'price_change']].describe()

## Train/Test Split

In [ ]:
# Define features and target
features = ['price', 'lag_m1', 'lag_m2', 'ma_3', 'price_change']
X = df_product1[features]
y = df_product1['sales']

# 80/20 time-based split
split_idx = int(0.8 * len(df_product1))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {X_train_scaled.shape[0]}")
print(f"Test set size: {X_test_scaled.shape[0]}")

## Model Comparison

In [ ]:
# Fit four models: OLS, Lasso, Ridge, ElasticNet
models = {
    'OLS': LinearRegression(),
    'Lasso': Lasso(alpha=1.0, max_iter=10000),
    'Ridge': Ridge(alpha=1.0),
    'ElasticNet': ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000)
}

results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results.append({'Model': name, 'MAE': mae, 'RMSE': rmse, 'R²': r2})
    models[name].predictions = y_pred

results_df = pd.DataFrame(results)
results_df

## Coefficient Comparison

In [ ]:
# Extract coefficients from each model
coef_data = {
    'OLS': models['OLS'].coef_,
    'Lasso': models['Lasso'].coef_,
    'Ridge': models['Ridge'].coef_,
    'ElasticNet': models['ElasticNet'].coef_
}

# Create grouped bar chart
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(features))
width = 0.2

for idx, (model_name, coefs) in enumerate(coef_data.items()):
    ax.bar(x + idx*width, coefs, width, label=model_name)

ax.set_xlabel('Features')
ax.set_ylabel('Coefficient Value')
ax.set_title('Coefficient Comparison Across Models')
ax.set_xticks(x + 1.5*width)
ax.set_xticklabels(features)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Lasso Regularization Path

In [ ]:
# Test Lasso across different alpha values
alphas = [0.01, 0.1, 1.0, 10.0, 100.0]
lasso_results = []

for alpha in alphas:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_train_scaled, y_train)
    y_pred = lasso.predict(X_test_scaled)
    
    mae = mean_absolute_error(y_test, y_pred)
    non_zero_coefs = np.sum(lasso.coef_ != 0)
    
    lasso_results.append({
        'Alpha': alpha,
        'MAE': mae,
        'Non-Zero Coefficients': non_zero_coefs
    })

lasso_df = pd.DataFrame(lasso_results)

# Plot Lasso regularization path
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.loglog(lasso_df['Alpha'], lasso_df['MAE'], marker='o', linewidth=2, markersize=8)
ax1.set_xlabel('Alpha (log scale)')
ax1.set_ylabel('MAE')
ax1.set_title('Lasso: Alpha vs MAE')
ax1.grid(True, alpha=0.3)

ax2.loglog(lasso_df['Alpha'], lasso_df['Non-Zero Coefficients'], marker='s', linewidth=2, markersize=8, color='orange')
ax2.set_xlabel('Alpha (log scale)')
ax2.set_ylabel('Number of Non-Zero Coefficients')
ax2.set_title('Lasso: Alpha vs Sparsity')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Ridge Regularization Path

In [ ]:
# Test Ridge across different alpha values
ridge_results = []

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_scaled, y_train)
    y_pred = ridge.predict(X_test_scaled)
    
    mae = mean_absolute_error(y_test, y_pred)
    ridge_results.append({'Alpha': alpha, 'MAE': mae})

ridge_df = pd.DataFrame(ridge_results)

# Plot Ridge regularization path
plt.figure(figsize=(8, 5))
plt.loglog(ridge_df['Alpha'], ridge_df['MAE'], marker='o', linewidth=2, markersize=8, color='green')
plt.xlabel('Alpha (log scale)')
plt.ylabel('MAE')
plt.title('Ridge: Alpha vs MAE')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Predictions vs Actuals

In [ ]:
# Use best-performing model from model comparison (lowest MAE)
best_model_name = results_df.loc[results_df['MAE'].idxmin(), 'Model']
best_model = models[best_model_name]
best_pred = best_model.predictions

# Scatter plot of actual vs predicted
plt.figure(figsize=(8, 6))
plt.scatter(y_test, best_pred, alpha=0.6, s=50)

# Add 45-degree reference line
min_val = min(y_test.min(), best_pred.min())
max_val = max(y_test.max(), best_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')

plt.xlabel('Actual Sales')
plt.ylabel('Predicted Sales')
plt.title(f'Actual vs Predicted Sales ({best_model_name})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Residual Analysis

In [ ]:
# Calculate residuals for best model
residuals = y_test - best_pred

# Histogram of residuals
plt.figure(figsize=(8, 5))
plt.hist(residuals, bins=20, edgecolor='black', alpha=0.7)
plt.xlabel('Residuals')
plt.ylabel('Frequency')
plt.title(f'Distribution of Prediction Errors ({best_model_name})')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()